# Electron Spin Resonance (ESR)
## Atomic and Molecular Physics Laboratory - Experiment 05

This notebook contains the complete data processing, numerical computation, error estimation, and artifact generation (plots and tables) for Experiment 05: Electron Spin Resonance (ESR).

### Objectives:
1. Determination of the Landé $g$-factor of the free-radical DPPH (Diphenylpicrylhydrazyl) sample.
2. Determination of the Full Width at Half Maximum (FWHM) absorption line width $\Delta B$.
3. Evaluation of experimental uncertainties against literature standards.

In [1]:
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fitz
from PIL import Image

os.makedirs('../plots', exist_ok=True)
os.makedirs('../report/tables', exist_ok=True)

# extract Figure 2 setup image from manual pdf if available
pdf_path = '../ATOMIC AND MOLECULAR PHYSICS LABORATORY_5th_.pdf'
if os.path.exists(pdf_path):
    doc = fitz.open(pdf_path)
    base_image = doc.extract_image(119)
    setup_img = Image.open(io.BytesIO(base_image['image']))
    setup_img.save('../plots/esr_setup.png')
    print("Extracted Figure 2 setup image to ../plots/esr_setup.png")

Extracted Figure 2 setup image to ../plots/esr_setup.png


### 1. Fundamental Constants and Calibration Parameters
Physical constants from CODATA and calibrated experimental parameters from the laboratory manual.

In [2]:
# si physical constants
h_planck = 6.62607015e-34 # j*s
h_manual = 6.63e-34 # j*s
mu_b = 9.274010078e-24 # j/t
mu_b_manual = 9.27e-24 # j/t
mu_0 = 4.0 * np.pi * 1e-7 # t*m/a

# experimental apparatus parameters
f_rf = 146.0e6 # hz
n_turns = 241 # coil turns
r_coil = 0.048 # m, 4.8 cm
v_operating = 10.0 # v, dc operating voltage

# coil geometry factors
k_b_ideal = (4.0 / 5.0)**1.5 * mu_0 * n_turns / r_coil # ideal helmholtz factor (t/a)
k_b_manual = 4.07e-3 # t/a, calibrated phywe geometry factor
g_const_manual = 2.565 # a, empirical factor from manual: (h*f/mu_b) / k_b

# literature reference values
g_lit = 2.0037 # dpph literature g-factor
g_free = 2.0023193 # free electron g-factor
delta_b_lit = 2.8e-4 # t, dpph resonance linewidth (2.8 x 10^-4 t)

print(f"Ideal Helmholtz factor: {k_b_ideal:.6e} T/A")
print(f"Calibrated coil factor: {k_b_manual:.6e} T/A")
print(f"Manual Lande constant: {g_const_manual:.4f} A")

Ideal Helmholtz factor: 4.514614e-03 T/A
Calibrated coil factor: 4.070000e-03 T/A
Manual Lande constant: 2.5650 A


### 2. Landé $g$-Factor Determination Across Trials
Calculation of the resonant magnetic field $B_r$ and Landé $g$-factor for each repeated measurement trial.

In [3]:
# load raw trial measurements
df_trials = pd.read_csv('../DATA/data.csv')

# compute resonant magnetic field and lande g-factor
df_trials['B_r_mT'] = df_trials['I_r_A'] * k_b_manual * 1e3
df_trials['g_calc'] = g_const_manual / df_trials['I_r_A']
df_trials['error_lit_pct'] = np.abs(df_trials['g_calc'] - g_lit) / g_lit * 100.0

# statistical averages
i_r_mean = df_trials['I_r_A'].mean()
i_r_std = df_trials['I_r_A'].std(ddof=1)
i_r_se = i_r_std / np.sqrt(len(df_trials))

b_r_mean = df_trials['B_r_mT'].mean()
b_r_std = df_trials['B_r_mT'].std(ddof=1)
b_r_se = b_r_std / np.sqrt(len(df_trials))

g_mean = df_trials['g_calc'].mean()
g_std = df_trials['g_calc'].std(ddof=1)
g_se = g_std / np.sqrt(len(df_trials))

g_rel_err = np.abs(g_mean - g_lit) / g_lit * 100.0
g_free_err = np.abs(g_mean - g_free) / g_free * 100.0

print(f"Mean Resonant Current: {i_r_mean:.3f} +/- {i_r_se:.4f} A (s = {i_r_std:.4f} A)")
print(f"Mean Resonant Field: {b_r_mean:.4f} +/- {b_r_se:.4f} mT")
print(f"Mean Lande g-factor: {g_mean:.5f} +/- {g_se:.5f} (s = {g_std:.5f})")
print(f"Relative error vs DPPH literature ({g_lit}): {g_rel_err:.2f}%")
print(f"Relative error vs free electron ({g_free:.4f}): {g_free_err:.2f}%")
display(df_trials)

Mean Resonant Current: 1.250 +/- 0.0058 A (s = 0.0100 A)
Mean Resonant Field: 5.0875 +/- 0.0235 mT
Mean Lande g-factor: 2.05209 +/- 0.00948 (s = 0.01642)
Relative error vs DPPH literature (2.0037): 2.41%
Relative error vs free electron (2.0023): 2.49%


,Trial,I_r_A,g_recorded,B_r_mT,g_calc,error_lit_pct
0,1,1.26,2.03571,5.1282,2.035714,1.597758
1,2,1.24,2.06800,5.0468,2.068548,3.236432
2,3,1.25,2.05200,5.0875,2.052000,2.410541


### 3. Full Width at Half Maximum (FWHM) Analysis
Determination of the absorption peak width $\Delta B$ using static DC displacement between half-maximum points.

In [4]:
# load raw fwhm measurements
df_fwhm = pd.read_csv('../DATA/data_fwhm.csv')
param_map = dict(zip(df_fwhm['parameter'].str.lower(), df_fwhm['value']))

i1_val = float(param_map['i1'])
i2_val = float(param_map['i2'])
delta_i = np.abs(i1_val - i2_val)
delta_b_calc = k_b_manual * delta_i
delta_b_lit_val = float(param_map['delta_b_lit'])
fwhm_err_pct = np.abs(delta_b_calc - delta_b_lit_val) / delta_b_lit_val * 100.0

print(f"Intersection current I1: {i1_val:.2f} A")
print(f"Intersection current I2: {i2_val:.2f} A")
print(f"Delta I: {delta_i:.2f} A")
print(f"Experimental FWHM Delta B: {delta_b_calc:.6e} T ({delta_b_calc * 1e3:.4f} mT)")
print(f"Literature FWHM Delta B: {delta_b_lit_val:.6e} T ({delta_b_lit_val * 1e3:.4f} mT)")
print(f"FWHM relative error: {fwhm_err_pct:.2f}%")

Intersection current I1: 0.15 A
Intersection current I2: 0.09 A
Delta I: 0.06 A
Experimental FWHM Delta B: 2.442000e-04 T (0.2442 mT)
Literature FWHM Delta B: 2.800000e-04 T (0.2800 mT)
FWHM relative error: 12.79%


### 4. High-Resolution Figure Generation
Exporting publication-grade graphics directly to `../plots/`.

In [5]:
# Figure 1: ESR absorption lineshape and FWHM determination
plt.figure(figsize=(7, 4.8), dpi=300)

gamma_hwhm = (delta_b_calc * 1e3) / 2.0 # hwhm in mt
b_range = np.linspace(b_r_mean - 0.75, b_r_mean + 0.75, 1000)

# lorentzian resonance absorption power profile
p_abs = 1.0 / (1.0 + ((b_range - b_r_mean) / gamma_hwhm)**2)

# plot resonance profile
plt.plot(b_range, p_abs, color='#1f77b4', lw=2.2, label='ESR Absorption Resonance Curve')

# half maximum level
plt.axhline(0.5, color='#d62728', linestyle='--', lw=1.5, label='Half-Maximum Level ($P / P_{max} = 0.5$)')

# experimental half power points
b_half_1 = b_r_mean - gamma_hwhm
b_half_2 = b_r_mean + gamma_hwhm
plt.scatter([b_half_1, b_half_2], [0.5, 0.5], color='#d62728', zorder=5, s=60, edgecolors='black', label=f'Half-Power Intersections ($\\Delta I = {delta_i:.2f}$ A)')

# annotate fwhm width
plt.annotate('', xy=(b_half_1, 0.5), xytext=(b_half_2, 0.5),
             arrowprops=dict(arrowstyle='<->', color='#2ca02c', lw=2.0))
plt.text(b_r_mean, 0.54, f'FWHM $\\Delta B = {delta_b_calc * 1e3:.3f}$ mT\n($\\Delta B_{{lit}} = {delta_b_lit_val * 1e3:.2f}$ mT)',
         ha='center', va='bottom', fontsize=10, color='#1b5e20', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#e8f5e9', edgecolor='#81c784', alpha=0.9))

# resonance center line
plt.axvline(b_r_mean, color='#555555', linestyle=':', lw=1.2, label=f'Resonant Center $B_r = {b_r_mean:.3f}$ mT')

plt.title('DPPH Electron Spin Resonance Absorption Signal & FWHM', fontsize=12, pad=12, fontweight='bold')
plt.xlabel('Magnetic Induction $B$ (mT)', fontsize=11)
plt.ylabel('Normalized RF Absorption Power $P / P_{max}$', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower center', fontsize=9, framealpha=0.95)
plt.ylim(0, 1.12)
plt.tight_layout()
plt.savefig('../plots/esr_absorption_fwhm.png')
plt.close()

In [6]:
# Figure 2: Lande g-factor comparison across experimental trials
plt.figure(figsize=(7, 4.5), dpi=300)

trial_indices = df_trials['Trial'].values
g_values = df_trials['g_calc'].values

# individual trial points
plt.errorbar(trial_indices, g_values, yerr=g_se, fmt='o', color='#1f77b4',
             ecolor='#1f77b4', elinewidth=2, capsize=6, capthick=1.5, markersize=8,
             label=f'Trial Values (Mean = {g_mean:.4f} $\\pm$ {g_se:.4f})', zorder=4)

# mean experimental line and confidence band
plt.axhline(g_mean, color='#1f77b4', linestyle='-', lw=1.6, alpha=0.8)
plt.fill_between([0.5, 3.5], g_mean - g_se, g_mean + g_se, color='#1f77b4', alpha=0.15, label='$\\pm 1\\sigma_{mean}$ Standard Error Band')

# reference values
plt.axhline(g_lit, color='#d62728', linestyle='--', lw=2.0, label=f'DPPH Literature ($g = {g_lit:.4f}$)')
plt.axhline(g_free, color='#2ca02c', linestyle='-.', lw=1.8, label=f'Free Electron ($g_e = {g_free:.4f}$)')

plt.title('Comparison of Landé $g$-Factor Across Trials', fontsize=12, pad=12, fontweight='bold')
plt.xlabel('Trial Number', fontsize=11)
plt.ylabel('Landé $g$-Factor', fontsize=11)
plt.xticks([1, 2, 3], ['Trial 1', 'Trial 2', 'Trial 3'], fontsize=10)
plt.xlim(0.6, 3.4)
plt.ylim(1.98, 2.10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', fontsize=9.5, framealpha=0.95)
plt.tight_layout()
plt.savefig('../plots/lande_g_trials.png')
plt.close()

In [7]:
# Figure 3: Zeeman energy splitting and RF magnetic resonance
plt.figure(figsize=(7, 4.8), dpi=300)

b_field = np.linspace(0, 8.0, 500) # mt
# convert to tesla for energy calculation
b_field_t = b_field * 1e-3
e_plus = +0.5 * g_mean * mu_b * b_field_t / 1.602176634e-19 * 1e6 # in micro-ev
e_minus = -0.5 * g_mean * mu_b * b_field_t / 1.602176634e-19 * 1e6 # in micro-ev

plt.plot(b_field, e_plus, color='#1f77b4', lw=2.2, label='$m_s = +1/2$ (Anti-parallel)')
plt.plot(b_field, e_minus, color='#ff7f0e', lw=2.2, label='$m_s = -1/2$ (Parallel)')

# resonance marker at resonant field
delta_e_rf = h_planck * f_rf / 1.602176634e-19 * 1e6 # in micro-ev
e_res_plus = +0.5 * delta_e_rf
e_res_minus = -0.5 * delta_e_rf

plt.axvline(b_r_mean, color='#555555', linestyle=':', lw=1.2, label=f'Resonant Field $B_r = {b_r_mean:.3f}$ mT')
plt.annotate('', xy=(b_r_mean, e_res_plus), xytext=(b_r_mean, e_res_minus),
             arrowprops=dict(arrowstyle='<->', color='#d62728', lw=2.2))
plt.scatter([b_r_mean, b_r_mean], [e_res_minus, e_res_plus], color='#d62728', zorder=5, s=50)

plt.text(b_r_mean + 0.25, 0, f'Resonance Condition:\n$h f = \\Delta E = {delta_e_rf:.4f}\\ \\mu\\mathrm{{eV}}$\n$f = 146.0$ MHz',
         va='center', fontsize=9.5, color='#9c0006', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffc7ce', edgecolor='#d62728', alpha=0.85))

plt.title('Zeeman Level Splitting and ESR Magnetic Resonance', fontsize=12, pad=12, fontweight='bold')
plt.xlabel('Magnetic Induction $B$ (mT)', fontsize=11)
plt.ylabel('Zeeman Energy Shift $\\Delta E$ ($\\mu\\mathrm{eV}$)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper left', fontsize=9.5, framealpha=0.95)
plt.xlim(0, 8.0)
plt.tight_layout()
plt.savefig('../plots/zeeman_energy_splitting.png')
plt.close()

### 5. Table Artifact Exports for LaTeX Integration
Exporting processed numeric tables into `../report/tables/`.

In [8]:
# Table 1: Lande g-factor trials export
table1_export = pd.DataFrame({
    'Trial': [str(x) for x in df_trials['Trial'].values] + ['Mean'],
    'I_r_A': [f"{x:.2f}" for x in df_trials['I_r_A'].values] + [f"{i_r_mean:.2f}"],
    'B_r_mT': [f"{x:.3f}" for x in df_trials['B_r_mT'].values] + [f"{b_r_mean:.3f}"],
    'g_calc': [f"{x:.4f}" for x in df_trials['g_calc'].values] + [f"{g_mean:.4f}"],
    'Error_pct': [f"{x:.2f}" for x in df_trials['error_lit_pct'].values] + [f"{g_rel_err:.2f}"]
})
table1_export.to_csv('../report/tables/table1_lande_g_trials.csv', index=False)
print("Table 1 exported successfully:")
display(table1_export)

# Table 2: FWHM results export
table2_export = pd.DataFrame({
    'Parameter': [
        'Right Intersection Current ($I_1$)',
        'Left Intersection Current ($I_2$)',
        'Current Line Width ($\Delta I$)',
        'Experimental FWHM ($\Delta B$)',
        'Literature FWHM ($\Delta B_{\\text{lit}}$)',
        'Relative Error'
    ],
    'Value': [
        f"{i1_val:.2f}",
        f"{i2_val:.2f}",
        f"{delta_i:.2f}",
        f"{delta_b_calc * 1e3:.4f}",
        f"{delta_b_lit_val * 1e3:.4f}",
        f"{fwhm_err_pct:.2f}"
    ],
    'Unit': ['A', 'A', 'A', 'mT', 'mT', '\%']
})
table2_export.to_csv('../report/tables/table2_fwhm_results.csv', index=False)
print("\nTable 2 exported successfully:")
display(table2_export)

# Table 3: Overall comparison summary
table3_export = pd.DataFrame({
    'Quantity': [
        'Lande $g$-Factor',
        'Resonant Magnetic Field $B_r$',
        'Absorption Linewidth $\Delta B$'
    ],
    'Experimental': [
        f"{g_mean:.4f} $\\pm$ {g_se:.4f}",
        f"{b_r_mean:.3f} mT",
        f"{delta_b_calc * 1e3:.3f} mT"
    ],
    'Literature': [
        f"{g_lit:.4f}",
        f"{10.43e-3 / g_lit * 1e3:.3f} mT",
        f"{delta_b_lit_val * 1e3:.3f} mT"
    ],
    'Rel_Error_pct': [
        f"{g_rel_err:.2f}",
        f"{np.abs(b_r_mean - (10.43e-3 / g_lit * 1e3)) / (10.43e-3 / g_lit * 1e3) * 100.0:.2f}",
        f"{fwhm_err_pct:.2f}"
    ]
})
table3_export.to_csv('../report/tables/table3_comparison_summary.csv', index=False)
print("\nTable 3 exported successfully:")
display(table3_export)

Table 1 exported successfully:


,Trial,I_r_A,B_r_mT,g_calc,Error_pct
0,1,1.26,5.128,2.0357,1.60
1,2,1.24,5.047,2.0685,3.24
2,3,1.25,5.088,2.0520,2.41
3,Mean,1.25,5.087,2.0521,2.41



Table 2 exported successfully:


,Parameter,Value,Unit
0,Right Intersection Current ($I_1$),0.15,A
1,Left Intersection Current ($I_2$),0.09,A
2,Current Line Width ($\Delta I$),0.06,A
3,Experimental FWHM ($\Delta B$),0.2442,mT
4,Literature FWHM ($\Delta B_{\text{lit}}$),0.2800,mT
5,Relative Error,12.79,\%



Table 3 exported successfully:


,Quantity,Experimental,Literature,Rel_Error_pct
0,Lande $g$-Factor,2.0521 $\pm$ 0.0095,2.0037,2.41
1,Resonant Magnetic Field $B_r$,5.087 mT,5.205 mT,2.26
2,Absorption Linewidth $\Delta B$,0.244 mT,0.280 mT,12.79
